# Model D: CRNN (CNN + BiLSTM + CTC)

Pre-trains on public synthetic words, then fine-tunes on the train split of my crops. Validation
during fine-tuning uses held-out train photos; the test split is never touched.

Drive has little space, so the full public sets are only ever downloaded and unpacked on the Colab
disk (`/content`). A random subset of `SUBSET` words is zipped to `data/public/<set>_subset.zip` on
Drive the first time; later sessions just unzip that. Training reads everything from `/content`.

Saves `models/crnn_<script>.pth` and `.json` (plus a `_pretrained` copy). Runtime: T4 GPU.

**Run the download and subset cells on a CPU runtime first.** They need no GPU and can take a long
time. Run the cells from the top down to the Kannada (or English) subset cell, until the
`<set>_subset.zip` is on Drive. Then switch to a T4 GPU runtime and run everything from the top. The
first data cell finds the zip on Drive and unzips it, and the download cells skip themselves.

In [ ]:
# Colab only
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ocr-project
!pip -q install jiwer

In [ ]:
import os
import sys
sys.path.insert(0, 'src')  # first on the path, so our datasets.py wins over the Hugging Face package
import torch
from torch.utils.data import DataLoader
from datasets import (WordDataset, alphabet, collate_words, crop_train_val, encodable, load_charset,
                      load_config, load_public, split_list, subset)
from models import CRNN, crnn_step, fit, load_model, save_model, set_seed

SCRIPT = 'kannada'  # or 'english'
SUBSET = 200000     # public words kept; only this subset is stored on Drive
REUSE_PRETRAINED = True  # fine-tune models/crnn_<script>_pretrained.pth if it exists, instead of pre-training again
WORKERS = 2
cfg = load_config()
public = 'mjsynth' if SCRIPT == 'english' else 'kannada_synth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

## Public words on the Colab disk

In [ ]:
# Colab only: reuse the subset saved on Drive by an earlier session, if there is one
drive_zip = f'data/public/{public}_subset.zip'
os.makedirs('data/public', exist_ok=True)
if os.path.exists(drive_zip):
    !unzip -qo {drive_zip} -d /content
    print('unzipped', drive_zip, 'to /content; skip the next three cells')
else:
    print(drive_zip, 'not on Drive yet; run the next cells for', SCRIPT)

In [ ]:
# Colab only, English, first time: stream MJSynth and keep SUBSET images (the 10 GB archive is never stored)
MJSYNTH_URL = ''  # link to mjsynth.tar.gz from https://www.robots.ox.ac.uk/~vgg/data/text/
if SCRIPT == 'english' and not os.path.exists(drive_zip):
    !python src/mjsynth_subset.py --url "{MJSYNTH_URL}" --out /content/mjsynth/90kDICT32px --n {SUBSET}
    !cd /content && zip -qr /content/drive/MyDrive/ocr-project/{drive_zip} mjsynth

In [ ]:
# Colab only, Kannada, first time: download the zip to the Colab disk (a CPU runtime is enough)
KANNADA_URL = 'https://ilocr.iiit.ac.in/public/scenetext/phase-0/v0.5/kannada/IIIT-Synthetic-Kannada/train.zip'
if SCRIPT == 'kannada' and not os.path.exists(drive_zip):
    # about 4.6 GB; -c continues a broken download, but only while this runtime lives
    !wget -c --progress=bar:force:noscroll -O /content/kannada_synth_train.zip "{KANNADA_URL}"
    # only the gt file and the vocabulary are unpacked, not the 2 million images
    !unzip -o -q /content/kannada_synth_train.zip train/train_gt.txt train/vocabulary.txt -d /content/kannada_raw
    !wc -l /content/kannada_raw/train/train_gt.txt /content/kannada_raw/train/vocabulary.txt

The zip holds `train/images/`, `train/train_gt.txt` and `train/vocabulary.txt`. The gt lines read `images/<name>.jpeg<TAB>word`, relative to the `train` folder inside the zip. The next cell picks `SUBSET` random words and extracts only their images straight from the zip.

In [ ]:
# Colab only, Kannada, first time: keep SUBSET random words and save only those to Drive
RAW_ZIP = '/content/kannada_synth_train.zip'
RAW_IMAGES = 'train'  # the folder inside the zip that the gt image names are relative to
RAW_GT = '/content/kannada_raw/train/train_gt.txt'
RAW_VOCAB = '/content/kannada_raw/train/vocabulary.txt'
if SCRIPT == 'kannada' and not os.path.exists(drive_zip):
    !python src/subset_words.py --zip "{RAW_ZIP}" --images "{RAW_IMAGES}" --gt "{RAW_GT}" --vocab "{RAW_VOCAB}" --out /content/kannada_synth --n {SUBSET}
    !cd /content && zip -qr /content/drive/MyDrive/ocr-project/{drive_zip} kannada_synth
    !mkdir -p data/public/kannada_synth && cp /content/kannada_synth/vocabulary.txt data/public/kannada_synth/

In [ ]:
# Colab only: training reads the public words and my crops from the Colab disk
cfg['public']['mjsynth']['root'] = '/content/mjsynth/90kDICT32px'
cfg['public']['kannada_synth']['train'] = {'images': '/content/kannada_synth/images',
                                           'gt': '/content/kannada_synth/train_gt.txt'}
if os.path.exists('data/crops/labels.tsv'):
    !cp -r data/crops /content/
    cfg['paths']['crops'] = '/content/crops'

## Data

In [ ]:
set_seed(cfg['seed'])
t = cfg['train']
alpha = alphabet(load_charset(cfg['crnn_alphabets'][SCRIPT]))  # code points, separate from the A-C classes
pretrained_path = os.path.join(cfg['paths']['models'], f'crnn_{SCRIPT}_pretrained.pth')
REUSE = REUSE_PRETRAINED and os.path.exists(pretrained_path)
# the public words are only needed when pre-training runs
words = [] if REUSE else encodable(load_public(public, 'train', cfg, SUBSET), alpha)
pre_train = split_list(words, 'train', 1 - t['pretrain_val'], cfg['seed'])
pre_val = split_list(words, 'test', 1 - t['pretrain_val'], cfg['seed'])
# before my photos are annotated there is no labels.tsv: then D is only pre-trained
FINETUNE = os.path.exists(os.path.join(cfg['paths']['crops'], 'labels.tsv'))
if FINETUNE:
    mine_train, mine_val = crop_train_val(cfg['paths']['crops'], SCRIPT, t['finetune_val'], cfg['seed'])
    mine_train, mine_val = encodable(mine_train, alpha), encodable(mine_val, alpha)
else:
    mine_train, mine_val = [], []
    print('no data/crops/labels.tsv yet: pre-training only, fine-tuning will be skipped')
print(f'{len(alpha)} symbols; pre-training {len(pre_train)} / {len(pre_val)} words, '
      f'fine-tuning {len(mine_train)} / {len(mine_val)} words')


def loader(samples, shuffle):
    data = WordDataset(samples, alpha, cfg['img_height'], t['crnn_max_width'])
    return DataLoader(data, batch_size=t['batch_size'], shuffle=shuffle, num_workers=WORKERS,
                      collate_fn=collate_words)

CTC needs at least as many output frames (image width / 4) as code points. Kannada packs several
code points into one akshara, so this counts the words that are too narrow. They give no gradient
(`zero_infinity`); if there are many, tell me before training.

In [ ]:
check = WordDataset(subset(pre_train, 2000, cfg['seed']), alpha, cfg['img_height'], t['crnn_max_width'])
narrow = sum(1 for x, y in check if CRNN.output_lengths(x.shape[2]) < len(y))
print(f'{narrow} of {len(check)} sampled words have fewer frames than code points')

## Pre-training on public words

In [ ]:
step = lambda m, batch, decode: crnn_step(m, batch, device, alpha, decode)
if REUSE:
    model, info = load_model(pretrained_path)
    # compared with the CRNN alphabet file only; changes to the A-C class list do not matter here
    assert info['alphabet'] == alpha, (f"{cfg['crnn_alphabets'][SCRIPT]} differs from the pre-trained model's "
                                       'alphabet; set REUSE_PRETRAINED = False')
    model = model.to(device)
    pre = info['pretrain']
    print(f'reusing {pretrained_path} (pre-trained on {info["pretrain_words"]} words), no pre-training now')
else:
    model = CRNN(len(alpha)).to(device)
    pre = fit(model, step, loader(pre_train, True), loader(pre_val, False), t['lr'], t['max_epochs'], t['patience'])
    info = dict(kind='crnn', script=SCRIPT, alphabet=alpha, hidden=256, height=cfg['img_height'],
                max_width=t['crnn_max_width'], seed=cfg['seed'], pretrain_words=len(pre_train), pretrain=pre)
    os.makedirs(cfg['paths']['models'], exist_ok=True)
    save_model(model, info, pretrained_path)

## Fine-tuning on the train split of my crops

In [ ]:
if FINETUNE:
    fine = fit(model, step, loader(mine_train, True), loader(mine_val, False), t['finetune_lr'], t['max_epochs'], t['patience'])
    info.update(finetune_words=len(mine_train), finetune=fine)
    path = os.path.join(cfg['paths']['models'], f'crnn_{SCRIPT}.pth')
    save_model(model, info, path)
    n_params = sum(p.numel() for p in model.parameters())
    total = pre['train_seconds'] + fine['train_seconds']
    print(f'saved {path}: {os.path.getsize(path) / 1e6:.1f} MB, {n_params:,} parameters, '
          f'total training time {total / 60:.1f} min')
else:
    print(f'fine-tuning skipped (no labels.tsv); the pre-trained model is crnn_{SCRIPT}_pretrained.pth, '
          f'pre-training took {pre["train_seconds"] / 60:.1f} min')